In [1]:
import os
import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from rapidfuzz import fuzz, process

from scholarlm.utils import get_filenames_in_directory, get_foldernames_in_directory

INFO 12-10 10:11:50 [__init__.py:216] Automatically detected platform cuda.


### Load Data

In [3]:
main_directory = os.getenv("POND_PATH")
main_directory = "/projectnb/mcnet/kevin/pond-papers/"
data_directory = os.getenv("POND_DATA_PATH")
data_directory = "/projectnb/mcnet/kevin/pond-data/"
pdf_directory = os.getenv("POND_PDF_PATH")
text_directory = os.getenv("POND_TEXT_PATH")
image_directory = os.getenv("POND_IMAGE_PATH")

In [31]:
# Directory
with open(os.path.join(main_directory, "directory.json"), "r") as f:
    paper_info = json.load(f)

registered_titles = [entry['title'] for entry in paper_info.values()]
registered_titles.sort()

# Original dataset
pond_data = pd.read_csv(os.path.join(data_directory, "pond_data.csv"), encoding_errors='ignore')
pond_data = pond_data.loc[pond_data.title.isin(registered_titles)]
pond_df = pond_data.loc[:,['author', 'title', 'pondname', 'location', 'author_term',
            'max_depth_m', 'mean_surfacearea_m2', 'macrophytes_percentcover', 'ph', 'tn_ugpl', 'tp_ugpl', 'chla_ugpl']]
pond_df.columns = ['author', 'title', 'name', 'location', 'ecosystem',
            'max_depth', 'surface_area', 'vegetation_cover', 'ph', 'tn', 'tp', 'chla']
pond_df = pond_df.sort_values(by = ['title', 'name'])
pond_df = pond_df.reset_index(drop=True)

In [32]:
pond_df

,author,title,name,location,ecosystem,max_depth,surface_area,vegetation_cover,ph,tn,tp,chla
0,chopyk et al.,agricultural freshwater pond supports diverse ...,NaN,central maryland; usa,ponds,3.35,2600.0,NaN,7.78,NaN,NaN,NaN
1,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,analysis of biological indicators related to t...,cuibul cu lebede lake,danube delta,shallow lake,NaN,NaN,NaN,8.03,NaN,NaN,NaN
2,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,analysis of biological indicators related to t...,erenciuc lake,danube delta,shallow lake,NaN,NaN,NaN,7.60,NaN,NaN,NaN
3,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,analysis of biological indicators related to t...,isac lake,danube delta,shallow lake,NaN,NaN,NaN,7.75,NaN,NaN,NaN
4,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,analysis of biological indicators related to t...,uzlina lake,danube delta,shallow lake,NaN,NaN,NaN,7.19,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1322,gilbert; juan diego; de vicente; inmaculada; j...,zooplankton body size versus taxonomy in medit...,castillo,alto guadalquivir region of spain,mediterranean wetland,NaN,5000.0,NaN,8.30,1100.0,57.8,8.9
1323,gilbert; juan diego; de vicente; inmaculada; j...,zooplankton body size versus taxonomy in medit...,hituelo,alto guadalquivir region of spain,mediterranean wetland,NaN,42000.0,NaN,8.10,700.0,55.5,7.8
1324,gilbert; juan diego; de vicente; inmaculada; j...,zooplankton body size versus taxonomy in medit...,orcera,alto guadalquivir region of spain,mediterranean wetland,NaN,4000.0,NaN,8.10,1100.0,35.4,7.6
1325,gilbert; juan diego; de vicente; inmaculada; j...,zooplankton body size versus taxonomy in medit...,quinta,alto guadalquivir region of spain,mediterranean wetland,NaN,57000.0,NaN,8.50,1000.0,49.1,8.8


In [36]:
def process(
    result_df,
    conversion_table
):
    result_df = result_df.copy()

    # Drop rows without any measurements
    result_df = result_df.dropna(subset=['value'])
    result_df = result_df.reset_index(drop=True)

    # Convert units
    processed_values = [val for val in result_df.loc[:,'value']]
    for row_idx, row in result_df.iterrows():
        measurement_type = row['measurement']
        val = row['value']
        unit = row['units']
        if conversion_table.get(measurement_type) is not None:
            if conversion_table[measurement_type].get(unit) is not None:
                conversion_factor = conversion_table[measurement_type][unit]
                processed_values[row_idx] = val * conversion_factor
            else:
                processed_values[row_idx] = np.nan
        

    result_df['processed_value'] = processed_values
    result_df = result_df.dropna(subset=['processed_value'])
    result_df = result_df.reset_index(drop=True)

    # Drop all unit columns
    #result_df = result_df.drop(columns=["units"])

    # Drop exact duplicates
    result_df = result_df.drop_duplicates()

    # Reset index
    result_df = result_df.reset_index(drop=True)

    return result_df


conversion_table = {
    'max_depth': {"cm": 0.01, "feet": 0.3048, "km": 1000, "m": 1},
    'surface_area': {"km^2": 1e6, "ha": 1e4, "mi^2": 2.59e6, "m^2": 1},
    'vegetation_cover': {"percent": 0.01, "fraction": 1},
    'tn': {"mg/L": 1000, "µg/L": 1, "μmol/L": 14.01, "ppm": 1000, "ppb": 1},
    'tp': {"mg/L": 1000, "µg/L": 1, "μmol/L": 30.97, "ppm": 1000, "ppb": 1},
    'chla': {"mg/L": 1000, "µg/L": 1},
    #'ph': {},
    #'latitude': {},
    #'longitude': {}
}

In [37]:
with open("../data/12_10_25/pond_full_paper_gemma2.json", "r") as f:
    result_dict = json.load(f)

result_df = pd.DataFrame(result_dict)

# Process measurements
ignore_measurements = ['latitude', 'longitude'] # Ignoring these for now because they are not in the original dataset
result_df = result_df.loc[~result_df.measurement.isin(ignore_measurements)]
result_df['value'] = result_df['value'].str.replace(',', '')  # Remove commas from numbers
result_df['value'] = pd.to_numeric(result_df['value'], errors='coerce')
result_df = process(result_df, conversion_table)
result_df = result_df.reset_index(drop=True)

# Re-format dataframe
result_cols = ['author', 'title', 'name', 'location', 'ecosystem', 'measurement', 'processed_value']
result_df = result_df.loc[:, result_cols]
result_df = result_df.pivot_table(index=['title', 'author', 'name', 'location', 'ecosystem'], 
                         columns='measurement', 
                         values='processed_value',
                         aggfunc='first').reset_index()
result_df = result_df.sort_values(by = ['title', 'name'])
result_df = result_df.reset_index(drop=True)

In [39]:
result_df.describe()

measurement,chla,max_depth,ph,surface_area,tn,tp,vegetation_cover
count,298.000000,831.000000,498.000000,1.083000e+03,3.440000e+02,4.720000e+02,237.000000
mean,1579.528463,64.431261,8.347811,9.583686e+08,1.050391e+05,1.116842e+04,3.937644
std,15042.132132,945.513056,9.101996,1.680738e+10,7.321612e+05,8.462198e+04,52.463254
min,0.000000,-0.025789,0.040000,1.000000e-03,0.000000e+00,8.000000e-03,0.000000
25%,3.425000,1.000000,6.452500,1.406500e+03,4.452500e+02,1.935000e+01,0.080000
50%,11.600000,3.000000,7.370000,1.100000e+04,9.980000e+02,7.350000e+01,0.310000
75%,40.500000,12.750000,8.100000,1.210000e+05,2.960000e+03,3.916040e+02,0.680000
max,240000.000000,27000.000000,119.000000,3.800000e+11,1.165800e+07,1.020000e+06,808.000000


In [40]:
pond_df.describe()

,max_depth,surface_area,vegetation_cover,ph,tn,tp,chla
count,530.000000,983.000000,201.000000,559.000000,237.000000,473.000000,427.000000
mean,2.093172,23600.227707,45.335821,7.400993,2395.714768,7015.152400,52.023904
std,2.011758,38957.770996,35.823789,1.057596,3893.038105,35215.422751,144.210983
min,0.090000,0.900000,0.000000,3.800000,247.600000,0.124000,0.000000
25%,0.700000,1318.000000,13.000000,6.757500,655.000000,30.700000,4.550000
50%,1.200000,5540.000000,38.000000,7.600000,1100.000000,120.000000,15.400000
75%,3.000000,28000.000000,76.000000,8.100000,2270.000000,470.000000,39.975000
max,9.000000,193820.000000,100.000000,10.300000,31200.000000,490000.000000,1704.000000


In [35]:
result_df.columns

Index(['title', 'author', 'name', 'location', 'ecosystem', 'chla', 'max_depth',
       'surface_area', 'tn', 'tp', 'vegetation_cover'],
      dtype='object', name='measurement')